# 12장 실습 — 결과를 지표와 그림으로

시뮬레이션을 돌리는 것으로 일이 끝나지 않습니다.
결과를 읽고, 판단하고, 남에게 설명해야 합니다.
이 실습에서 만드는 표와 그림이 기말 프로젝트 보고서의 뼈대가 됩니다.
교재 12장에 대응합니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 지표 세 무리 (교재 12.1)

지표는 누구의 편인지에 따라 셋으로 나뉩니다.

| 무리 | 누구를 위한 것인가 | 예 |
|---|---|---|
| 승객 | 서비스를 쓰는 사람 | 서비스율, 평균 대기, 90퍼센타일 대기 |
| 차량 | 서비스를 운영하는 사람 | 가동률, 공차 주행거리 |
| 시스템 | 도시 전체 | 총 주행거리, 공차 비율 |

셋은 서로 당깁니다. 승객을 좋게 하면 차량이 놀고, 차량을 바쁘게 하면 승객이 기다립니다.

In [ ]:
from smartmob import Dtumos
from smartmob.data import load_demand, load_vehicles
from smartmob.teaching.metrics import kpi_table
from smartmob.teaching.simloop import simulate

demand = load_demand("hanam")
vehicles = load_vehicles("hanam")

engine = Dtumos().run_simulation(
    city="hanam", mode="taxi", fleet_size=80, num_passengers=1000,
    time_start=1080, time_end=1440, dispatch_mode="optimization",
    matrix_mode="street_distance", vehicle_capacity=1, random_seed=42,
)
mine = simulate(demand, vehicles, 1080, 1440)

`kpi_table` 은 엔진 결과와 직접 짠 결과를 **같은 표** 로 냅니다.
형식이 같아야 나란히 놓고 비교할 수 있습니다.

In [ ]:
import pandas as pd

pd.DataFrame({"내 루프": kpi_table(mine), "DTUMOS": kpi_table(engine)}).round(3)

## 2. 평균은 왜 위험한가 (교재 12.2)

1장에서 본 것을 다시 봅니다. 이번에는 보고서에 쓸 형태로 만듭니다.

In [ ]:
waits = [r.wait_min for r in mine.requests if r.pickup_time is not None]
row = pd.Series({
    "평균": pd.Series(waits).mean(),
    "중앙값": pd.Series(waits).median(),
    "80퍼센타일": pd.Series(waits).quantile(0.8),
    "90퍼센타일": pd.Series(waits).quantile(0.9),
    "최댓값": max(waits),
}).round(2)
row

보고서에는 평균 대신 **중앙값과 90퍼센타일** 을 씁니다.
"절반은 X분 안에 탔고, 열에 아홉은 Y분 안에 탔다"가 평균 한 줄보다 정확합니다.

## 3. 차량을 몇 대 둘 것인가 (교재 12.4)

이 과목에서 답해야 할 대표적인 질문입니다.
대수를 바꿔 가며 돌리고, 승객 지표와 차량 지표를 나란히 놓습니다.

In [ ]:
from smartmob.teaching.metrics import compare

runs = {}
for n in [20, 30, 40, 60, 80, 120, 160]:
    runs[f"{n}대"] = simulate(demand, vehicles.head(n), 1080, 1440)

table = compare(runs)
table.round(3)

## 4. 파레토 곡선

두 지표를 축으로 놓으면 "여기서 더 좋아지려면 저기를 포기해야 하는" 경계가 보입니다.

In [ ]:
import matplotlib.pyplot as plt

x = [runs[k].summary()["utilization"] for k in runs]
y = [runs[k].summary()["avg_waiting_time_min"] for k in runs]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x, y, "-o", color="#4C6EF5")
for label, xi, yi in zip(runs, x, y):
    ax.annotate(label, (xi, yi), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.set_xlabel("차량 가동률")
ax.set_ylabel("평균 대기시간 (분)")
ax.set_title("승객과 사업자는 반대 방향으로 당깁니다")
ax.grid(alpha=0.3)
plt.tight_layout();

곡선 위의 어느 점을 고를지는 계산이 아니라 판단입니다.
보고서에는 고른 점과 **고른 이유** 를 적습니다.

## 5. 시간대별로 보기 (교재 12.6)

하루 평균 한 줄에는 러시아워가 묻힙니다.

In [ ]:
record = mine.record.copy()
record["hour"] = record["time"] // 60
by_hour = record.groupby("hour")[
    ["waiting_passenger_cnt", "driving_vehicle_cnt", "empty_vehicle_cnt"]
].mean().round(1)
by_hour

In [ ]:
from smartmob.viz import plot_record

plot_record(mine.record);

## 6. 결과를 웹으로 내보내기

발표에서 보여 줄 것은 표가 아니라 움직이는 그림입니다.
DTUMOS 화면이 하는 일을 파일 하나짜리 뷰어로 다시 만듭니다.

먼저 데이터를 내보냅니다. 파일 이름과 키 이름은 DTUMOS 가 내는 것 그대로입니다.
그래서 서버에서 받은 결과 디렉터리를 그대로 넣어도 뷰어가 읽습니다.

In [ ]:
from smartmob.viz import export_viewer

export_viewer(engine, "ch12_viewer/public/data", sample=200)

직접 짠 루프의 결과도 같은 함수로 내보낼 수 있습니다.
그쪽은 도로망 위를 달리지 않으므로 통행이 직선 하나로 나옵니다.
그 직선이 곧 11장 모형의 근사 수준입니다. 나란히 보면 차이가 바로 보입니다.

In [ ]:
# 직접 짠 루프 결과로 바꿔 보려면 아래 주석을 풉니다.
# export_viewer(mine, "ch12_viewer_data")

이제 뷰어를 실행합니다. 터미널에서 다음을 실행합니다.
Node 20.19 이상이 필요합니다.

```bash
cd labs/ch12_viewer
npm install        # 처음 한 번만
npm run dev        # npm start 도 같습니다
```

브라우저가 자동으로 열립니다.

처음 열면 지도만 나오고 차가 움직이지 않습니다. 빈칸이 다섯 자리 있기 때문입니다.
화면 왼쪽에 무엇이 남았는지 뜹니다.
`src/main.js` 를 고치고 저장하면 브라우저가 알아서 다시 그립니다. 새로고침도 필요 없습니다.

| 빈칸 | 하는 일 |
|---|---|
| `tripPath` | 구간의 좌표열을 꺼냅니다 |
| `tripTimestamps` | 각 좌표의 시각을 꺼냅니다 |
| `tripColor` | 탑승과 공차의 색을 나눕니다 |
| `waitingPassengers` | 지금 기다리는 승객만 고릅니다 |
| `nextTime` | 한 프레임만큼 시간을 흘립니다 |

다섯 자리를 채우면 저녁 6시부터 자정까지 하남시에서 무슨 일이 있었는지가
화면에서 재생됩니다.

파이썬 대신 자바스크립트로 짜는 이유는 하나입니다.
지도 위에서 시간을 흘리는 일은 브라우저가 훨씬 잘합니다.
DTUMOS 프론트엔드도 같은 라이브러리로 같은 두 키를 읽습니다.

채운 뒤 채점은 저장소 뿌리에서 합니다.

```bash
node tools/check_viewer.mjs
```

발표에 쓸 주소가 필요하면 정적 파일로 만들어 깃허브 페이지에 올립니다.

```bash
npm run build      # dist/ 에 생깁니다
```

## 7. 빈칸

### 7.1 서비스 수준을 정합니다

"호출의 90%가 5분 안에 배차된다"를 목표로 잡습니다.
이 목표를 만족하는 최소 차량 대수를 위 스윕에서 찾습니다.
필요하면 대수를 더 촘촘히 넣어 다시 돌립니다.

In [ ]:
min_fleet = None        # 90퍼센타일 대기가 5분 이하가 되는 최소 차량 대수

banner("빈칸 7.1")
todo("필요한 최소 차량 대수", min_fleet)

### 7.2 공차 주행

차량이 승객 없이 달린 거리가 전체의 몇 퍼센트인지 구합니다.
대수를 늘리면 이 비율이 어떻게 되는지도 봅니다.
도시 전체로 보면 이것이 배출가스이자 혼잡입니다.

In [ ]:
empty_share_80 = None       # 80대일 때 공차 주행 비율 (0~1)
empty_share_160 = None      # 160대일 때 공차 주행 비율 (0~1)

banner("빈칸 7.2")
todo("80대 공차 비율", empty_share_80, fmt=lambda v: f"{v:.1%}")
todo("160대 공차 비율", empty_share_160, fmt=lambda v: f"{v:.1%}")

### 7.3 보고서 한 장

아래 다섯 줄을 채워 제출합니다. 기말 프로젝트 보고서의 결론이 이 형식입니다.

1. 무엇을 물었는가 (한 문장)
2. 어떻게 실험했는가 (수요, 차량, 시간 범위, seed)
3. 결과 표 한 개와 그림 한 장
4. 고른 답과 그 이유
5. 이 결과를 믿기 어렵게 만드는 것 한 가지

5번이 배점입니다. 자기 실험의 약점을 아는 것이 이 과목에서 배우는 것입니다.

In [ ]:
my_conclusion = None    # 4번을 한 문장으로

banner("빈칸 7.3")
todo("결론 한 문장", my_conclusion)

## 정리

- 지표는 승객·차량·시스템 셋으로 나뉘고 서로 당깁니다
- 평균 대신 중앙값과 90퍼센타일을 씁니다
- `compare()` 가 여러 시나리오를 한 표로 냅니다. `kpi_table` 덕분에 엔진 결과도 같은 표에 들어갑니다
- 파레토 곡선 위의 점을 고르는 것은 계산이 아니라 판단입니다
- `export_viewer` 가 낸 파일을 `ch12_viewer/` 프로젝트가 읽습니다. DTUMOS 와 같은 키를 씁니다
- 프로젝트에서는 여러분이 고른 시군구로 이 절차를 그대로 반복합니다